# LaserCut GmbH — Exploratory Data Analysis (EDA)

**Scope:** Only the assignment step: explore the already-extended dataset using suitable statistics and visualizations, investigate relationships and unusual observations.

This notebook does **not** extract features from DXF files. It assumes your CSV already contains the geometric features.

## Questions
1. What do the feature distributions look like?
2. Are there missing or suspicious values?
3. Which features vary the most?
4. Which features appear related?
5. Which unusual parts should be inspected in their DXF drawings?

Expected feature columns: `width`, `height`, `cutting_length`, `number_of_holes`, `bounding_box_area`.

## 1. Load the dataset

Place this notebook in your project folder, or change `DATASET_PATH` to your CSV location.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET_PATH = Path("laserCut_fake_test_data.csv")  # Change if needed
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_with_features.csv'

## 2. Select the features

Edit the names below if your CSV uses different column names. `ID_COLUMNS` are identifiers (such as filename) used to locate unusual parts later.

In [ ]:
FEATURES = [
    "width",
    "height",
    "cutting_length",
    "number_of_holes",
    "bounding_box_area",
]
ID_COLUMNS = ["filename", "customer"]  # Edit to match your CSV

available_features = [c for c in FEATURES if c in df.columns]
missing_features = [c for c in FEATURES if c not in df.columns]
print("Available features:", available_features)
print("Missing expected features:", missing_features)

if not available_features:
    raise ValueError("No expected feature columns found. Edit FEATURES to match your CSV.")

for col in available_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_df = df[available_features].copy()

## 3. Data quality and plausibility checks

Check data types, missing values, duplicates and negative measurements. Negative geometric measurements are usually suspicious and should be investigated.

In [ ]:
print("Data types:")
display(df[available_features].dtypes.to_frame("dtype"))

print("\nMissing values:")
display(pd.DataFrame({
    "missing_count": numeric_df.isna().sum(),
    "missing_percent": (numeric_df.isna().mean() * 100).round(2)
}))

print(f"\nDuplicate complete rows: {df.duplicated().sum()}")

print("\nNegative values (review these):")
display((numeric_df < 0).sum().to_frame("negative_count"))

if {"width", "height", "bounding_box_area"}.issubset(df.columns):
    expected_area = df["width"] * df["height"]
    difference = (df["bounding_box_area"] - expected_area).abs()
    print("\nBounding-box area mismatches (> 1e-6):", (difference > 1e-6).sum())

## 4. Descriptive statistics

`describe()` gives count, mean, standard deviation, minimum, quartiles and maximum.

- **Mean vs. median:** a large difference can indicate skewness or extreme values.
- **Standard deviation:** spread, expressed in the feature's own units.
- **Quartiles:** describe the middle 50% of observations.
- **Minimum and maximum:** help identify unusual values.

In [ ]:
summary = numeric_df.describe().T
summary["median"] = numeric_df.median()
summary["skewness"] = numeric_df.skew()
summary["missing"] = numeric_df.isna().sum()
display(summary.round(3))

## 5. Feature distributions: histograms

Histograms show how frequently values occur across ranges.

Look for right/left skew, multiple peaks, long tails and extreme values. The plots use original units.

In [ ]:
for feature in available_features:
    values = numeric_df[feature].dropna()
    if values.empty:
        continue
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(values, bins=30, edgecolor="black")
    ax.set(title=f"Distribution of {feature}", xlabel=feature, ylabel="Number of observations")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 6. Boxplots and unusual observations

A boxplot summarizes the median and quartiles and highlights potential outliers.

The IQR rule flags values below `Q1 - 1.5 × IQR` or above `Q3 + 1.5 × IQR`.

**An outlier is not automatically an error.** A very large or intricate part may be a genuine product. Inspect its DXF before deciding.

In [ ]:
for feature in available_features:
    values = numeric_df[feature].dropna()
    if values.empty:
        continue
    fig, ax = plt.subplots(figsize=(7, 2.8))
    ax.boxplot(values, vert=False)
    ax.set(title=f"Boxplot of {feature}", xlabel=feature)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

In [ ]:
outlier_rows = []

for feature in available_features:
    values = numeric_df[feature].dropna()
    if values.empty:
        continue
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = numeric_df[feature].notna() & (
        (numeric_df[feature] < lower) | (numeric_df[feature] > upper)
    )
    for idx in df.index[mask]:
        row = {
            "row_index": idx, "feature": feature, "value": df.loc[idx, feature],
            "lower_bound": lower, "upper_bound": upper
        }
        for id_col in ID_COLUMNS:
            if id_col in df.columns:
                row[id_col] = df.loc[idx, id_col]
        outlier_rows.append(row)

outliers = pd.DataFrame(outlier_rows)
if outliers.empty:
    print("No IQR outliers detected.")
else:
    display(outliers.sort_values(["feature", "value"], ascending=[True, False]))

## 7. Relationships: correlation matrix

Pearson correlation measures the strength and direction of **linear** relationships.

- Near `+1`: strong positive linear association.
- Near `-1`: strong negative linear association.
- Near `0`: weak linear association.

Correlation does not prove causation and can miss nonlinear relationships. Extreme observations can influence it.

In [ ]:
correlation = numeric_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(correlation.columns)))
ax.set_yticks(range(len(correlation.index)))
ax.set_xticklabels(correlation.columns, rotation=45, ha="right")
ax.set_yticklabels(correlation.index)
ax.set_title("Pearson correlation matrix")

for i in range(len(correlation.index)):
    for j in range(len(correlation.columns)):
        value = correlation.iloc[i, j]
        if pd.notna(value):
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=9)

fig.colorbar(im, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()
display(correlation.round(3))

## 8. Scatterplots: investigate engineering questions

Recommended comparisons:

1. **Bounding box area vs. cutting length:** do larger envelopes tend to have longer cutting paths?
2. **Number of holes vs. cutting length:** do parts with more holes tend to have longer cutting paths?
3. **Width vs. height:** are parts concentrated around particular proportions?
4. **Bounding box area vs. number of holes:** are larger parts also more perforated?

Look for trends, clusters, curved patterns and isolated points. A trend is an association, not proof of causation.

In [ ]:
recommended_pairs = [
    ("bounding_box_area", "cutting_length"),
    ("number_of_holes", "cutting_length"),
    ("width", "height"),
    ("bounding_box_area", "number_of_holes"),
]

for x_col, y_col in recommended_pairs:
    if x_col not in df.columns or y_col not in df.columns:
        print(f"Skipping {x_col} vs. {y_col}: column missing.")
        continue
    plot_data = df[[x_col, y_col]].dropna()
    if plot_data.empty:
        print(f"Skipping {x_col} vs. {y_col}: no complete observations.")
        continue
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.scatter(plot_data[x_col], plot_data[y_col], alpha=0.65)
    ax.set(title=f"{x_col} vs. {y_col}", xlabel=x_col, ylabel=y_col)
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 9. Retrieve the unusual rows for DXF inspection

For each unusual row:

1. Find its filename or part identifier.
2. Open the corresponding DXF.
3. Check whether the unusual value reflects a genuine design (large envelope, long contour, many internal features).
4. Check for extraction/data issues (wrong units, duplicate geometry, unsupported DXF entities).

In [ ]:
if not outliers.empty:
    unusual_indices = sorted(outliers["row_index"].unique())
    columns_to_show = [c for c in ID_COLUMNS + available_features if c in df.columns]
    display(df.loc[unusual_indices, columns_to_show])
else:
    print("There are no IQR-flagged rows to display.")

## 10. Findings and interpretation — complete after running the notebook

Use your actual outputs to answer:

### Distributions
- Which feature has the largest spread?
- Which features are skewed?
- Are there multiple clusters or peaks?

### Relationships
- Which pairs have the strongest correlations?
- Does bounding box area appear related to cutting length?
- Does hole count appear related to cutting length?
- Do scatterplots reveal clusters or nonlinear patterns?

### Unusual observations
- Which parts are flagged by the IQR rule?
- Are they genuine designs or possible data/extraction errors?
- What do their DXF drawings show?

### Manufacturing interpretation
- Cutting length can approximate laser travel distance, but is not a direct measure of cutting time.
- Hole count may indicate internal contours and piercing operations, depending on how holes are defined and detected.
- Bounding box dimensions describe the overall envelope, not actual material area or retrieval difficulty.
- Small or intricate parts may be difficult to retrieve even if their cutting length is not especially high.

### Limitations
Mention factors not represented by these geometric features, such as material, thickness, cutting speed, machine settings, piercing time, sheet layout/nesting and physical handling.

**Do not write conclusions before inspecting your actual outputs.** The notebook provides evidence; your interpretation should describe what that evidence supports.